In [6]:
from cresowlve.utils import read_json

data = read_json("../experiments/data/human_annot/chgk_no_ext_mat_ru_spec_fltrd_human_annot_split2.json")

In [8]:
from collections import defaultdict

annot_dist = defaultdict(int)

for sample in data["data"]:
    annot_dist[sample["annot_check"]] += 1

print(annot_dist)

defaultdict(<class 'int'>, {'1': 693, '4': 94, '3': 8, '2': 54, '6': 21, '7': 12, '0': 4, '5': 5, '9': 5, '': 1})


In [4]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()

response = client.responses.create(
    model="gpt-5.2",
    input="You are an expert at solving creative thinking puzzles. You are given a puzzle question that requires creative reasoning over real-world knowledge. Note that the question might contain several placeholder words (often in all capital case such as X, Y, THIS, FIRST, SECOND, ALPHA, BETA, HE, SHE, IT, HIM, HIS, HER, THEY, THEM etc.) that substitute for specific entities, objects, or concepts. These placeholders are crucial for solving the puzzle, and their meaning can only be inferred through careful reasoning about the question. Additionally, some placeholders might be gendered (e.g., HE/HIM/HIS vs SHE/HER), but do not assume that the gendered pronouns necessarily refer to human characters; they could refer to any entities, and their gender might be different. Please provide your final answer within <Answer>...</Answer> tags.\nPuzzle:\nIn a wartime novel, the hero looks at the sky, where contrails of airplanes and bursts of anti-aircraft shells form an unpleasant, in his opinion, picture. What word did we replace with \"picture\"?",
    reasoning={
        "effort": "medium",
        "summary": "auto"
    }
)

print(response.output)

[ResponseReasoningItem(id='rs_090fbd4b7d86c45c0069b1a27aadb8819794c4cee0af2c84a2', summary=[Summary(text='**Analyzing metaphorical imagery**\n\nI’m interpreting "contrails of airplanes and bursts of anti-aircraft shells" as imagery of high-altitude bombing and flak. Maybe it’s referencing a novel like "On the Western Front 1943," but I\'m unsure if it fits. The words hint at an unpleasant picture, perhaps something like "decorative" or "panorama." I’m considering how "lacework" could serve as a metaphor, showing how contrails and shell bursts create a nasty pattern in the sky. Another option is thinking of a "spiderweb" analogy. Ultimately, "embroidery" or "spiderweb" might be the strongest choices.', type='summary_text'), Summary(text='**Deciding on word replacements**\n\nThe question indicates that we need to replace "picture" with a synonym. In Russian, it could mean "картина" or "зрелище" for spectacle, but here "picture" likely translates to "картинка." I\'m thinking of potential 

In [ ]:
text_parts = []
thinking_parts = []
for item in getattr(response, "output", []):
    for c in getattr(item, "summary", []) + getattr(item, "content", []):
        t = getattr(c, "text", None)
        if not t and isinstance(c, dict):
            t = c.get("text")
        if t:
            ctype = getattr(c, "type", None)
            if ctype == "summary_text":
                thinking_parts.append(t)
            else:
                text_parts.append(t)

KeyboardInterrupt: 

In [3]:
len(response.output)

1

In [3]:
from dotenv import load_dotenv
load_dotenv()
from cresowlve.inference_lm import get_inference_service

prompt = "What is the sum of the first 5 prime numbers?"
service = get_inference_service("gemini-3.1-pro-preview")
response = await service.generate([{"id": "1", "user_prompt": prompt}], model_args={"enable_thinking_summary": True})

In [4]:
response

[ModelResponse(sample_id='1', text='', usage=None, exception=ServerError("503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}"), thoughts=None)]

In [6]:
google_model_args = service.prepare_model_args({"enable_thinking_summary": True})

In [7]:
from google.genai import types as genai_types
from google import genai

client = genai.Client()

config = genai_types.GenerateContentConfig(**google_model_args)
prompt = "What is the sum of the first 3 prime numbers?"

response = client.models.generate_content(
  model="gemini-3-flash-preview",
  contents=prompt,
  config=config
)

In [8]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""**Calculating the Sum of the First Three Prime Numbers**

Okay, so the target is the sum of the first three prime numbers. Let's break this down systematically. First, I need to define what a prime number *is*. Right, it's a whole number, definitely greater than one, and with only *two* factors: one and itself. Simple enough.

Now, let's find the first three. I'll go through the numbers sequentially, checking the primality of each:

*   One? Nope. Only has one factor.
*   Two? Yes! Factors are 1 and 2. That's prime number one.
*   Three? Yes again. Factors are 1 and 3. Prime number two.
*   Four? No, no. Factors are 1, 2, and 4.
*   Five? Yes! 1 and 5. That's prime number three.

So the first three prime numbers are 2, 3, and 5. Now, the sum. It is a straightforward addition: 2 + 3 + 5. I'll do this mentally. 2 + 3 = 

In [9]:
text_parts = []
thought_parts = []
for cand in getattr(response, "candidates", []) or []:
    content = getattr(cand, "content", None)
    if not content:
        continue
    for part in (getattr(content, "parts", []) or []):
        thought = getattr(part, "thought", None)
        txt = getattr(part, "text", None)
        if isinstance(txt, str) and txt.strip():
            if thought:
                thought_parts.append(txt.strip())
            else:
                text_parts.append(txt.strip())
if text_parts:
    text = "\n".join(text_parts).strip()
if thought_parts:
    thoughts = "\n".join(thought_parts).strip()

In [12]:
response.text

'The sum of the first 3 prime numbers is **10**.\n\nHere is the breakdown:\n1. The first prime number is **2**\n2. The second prime number is **3**\n3. The third prime number is **5**\n\n**2 + 3 + 5 = 10**'

In [11]:
thoughts

"**Calculating the Sum of the First Three Prime Numbers**\n\nOkay, so the target is the sum of the first three prime numbers. Let's break this down systematically. First, I need to define what a prime number *is*. Right, it's a whole number, definitely greater than one, and with only *two* factors: one and itself. Simple enough.\n\nNow, let's find the first three. I'll go through the numbers sequentially, checking the primality of each:\n\n*   One? Nope. Only has one factor.\n*   Two? Yes! Factors are 1 and 2. That's prime number one.\n*   Three? Yes again. Factors are 1 and 3. Prime number two.\n*   Four? No, no. Factors are 1, 2, and 4.\n*   Five? Yes! 1 and 5. That's prime number three.\n\nSo the first three prime numbers are 2, 3, and 5. Now, the sum. It is a straightforward addition: 2 + 3 + 5. I'll do this mentally. 2 + 3 = 5, and then 5 + 5 = 10. Done. The sum is 10."

In [11]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""**Finding the Sum of the First Three Primes**

Okay, so the goal here is straightforward: calculate the sum of the first three prime numbers. Let's break this down systematically.

First, I need to define what a prime number *is*. Right, it's a natural number greater than 1 that's only divisible by 1 and itself. Simple enough. Now I can list the primes in order, starting with 2, because that's the smallest prime. So the sequence begins: 2, 3, 5, 7, 11, and so on.

Next, I select the first three numbers from the list I've just written: 2, 3, and 5.

Now, let's sum these up. I need to calculate 2 + 3 + 5. I'll go step by step: first 2 + 3, which equals 5. Then I take that intermediate sum and add the next number in the sum: 5 + 5 which equals 10.

Therefore, the first three prime numbers are 2, 3, and 5, and their sum i

In [3]:
from cresowlve.inference_lm import get_inference_service

service = get_inference_service("gemini-3-flash-preview")

response = service.client.models.generate_content(model=service.model_name, contents="What is the capital of France?")

In [6]:
response.candidates[0].finish_reason == "STOP"

True

In [12]:
text_parts = []
thinking_parts = []
for item in getattr(response, "output", []):
    for c in getattr(item, "summary", []) + getattr(item, "content", []):
        t = getattr(c, "text", None)
        if not t and isinstance(c, dict):
            t = c.get("text")
        if t:
            ctype = getattr(c, "type", None)
            if ctype == "summary_text":
                thinking_parts.append(t)
            else:
                text_parts.append(t)
text = None
thoughts = None
if thinking_parts:
    thoughts = "\n".join(thinking_parts).strip()
if text_parts:
    text = "\n".join(text_parts).strip()

In [13]:
text, thoughts

('The capital of France is Paris.', None)

In [1]:
from datasets import load_dataset

muce = load_dataset("CNCL-Penn-State/MuCE", split="test")

/home/ismayilz/.conda/envs/cresowlve/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
prompts = set(muce["FullPrompt"])

In [7]:
len(prompts)

127

In [14]:
import json
with open("muce_test_prompts.json", "w") as f:
    f.write(json.dumps(list([p for p in prompts if p]), indent=2))

In [1]:
from cresowlve.utils import read_json

data1 = read_json("../experiments/data/error_analysis/samples/gemini_3_flash_preview_en0_sample1_ids.json")
data2 = read_json("../experiments/data/error_analysis/analysis.json")

In [3]:
new_data = [{"id": sample_id, "analysis": ""} for sample_id in data1["data"]]

for sample in data2:
    for new_sample in new_data:
        if new_sample["id"] == sample["id"]:
            new_sample["analysis"] = sample["analysis"]
            break

data1["data"] = new_data

In [5]:
from cresowlve.utils import write_json

write_json(data1, "../experiments/data/error_analysis/samples/gemini_3_flash_preview_en0_sample1_ids.json")

In [7]:
from cresowlve.utils import read_json, write_json

data = read_json("../experiments/data/error_analysis/samples/gemini_3_flash_preview_en0_sample3_analysis.json")

data["data"] = [{"id": sample_id, "analysis": ""} for sample_id in data["data"]]

write_json(data, "../experiments/data/error_analysis/samples/gemini_3_flash_preview_en0_sample3_analysis.json")

In [3]:
from cresowlve.utils import read_json, write_json

data = read_json("../experiments/data/error_analysis/samples/gemini_3_flash_preview_en0_sample2_analysis.json")
analysis_data = read_json("/Users/mismayil/Downloads/merged_output.json")

new_data = []

for sample in data["data"]:
    analysis_sample = next((a for a in analysis_data if a["id"] == sample["id"]), None)
    if analysis_sample:
        new_data.append({"id": sample["id"], "analysis": analysis_sample.get("my_comment", "")})

data["data"] = new_data
write_json(data, "../experiments/data/error_analysis/samples/gemini_3_flash_preview_en0_sample2_analysis.json")